# Load NHTSA complaints into Pinecone

Seeds the `carverdict` index with real vehicle-complaint data so the chat app has something to search. Run this once to populate the index.

In [1]:
import os, json, time, warnings
import requests
from dotenv import load_dotenv
from pinecone import Pinecone

warnings.filterwarnings("ignore")
load_dotenv(".env.local")

GEMINI_KEY = os.environ["GEMINI_API_KEY"]
PINECONE_KEY = os.environ["PINECONE_API_KEY"]
INDEX_NAME = os.environ["PINECONE_INDEX"]

/Users/sufiyanrauf/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
# A few popular vehicles to start the knowledge base.
# NHTSA needs the exact model name ("Silverado 1500", not "Silverado").
vehicles = [
    ("Toyota", "Camry", 2019),
    ("Honda", "Accord", 2019),
    ("Tesla", "Model 3", 2021),
    ("Ford", "Explorer", 2020),
    ("Chevrolet", "Silverado 1500", 2020),
]
per_vehicle = 15

def get_complaints(make, model, year):
    url = "https://api.nhtsa.gov/complaints/complaintsByVehicle"
    r = requests.get(url, params={"make": make, "model": model, "modelYear": year}, timeout=30)
    r.raise_for_status()
    return r.json().get("results", [])

records = []
for make, model, year in vehicles:
    results = get_complaints(make, model, year)
    print(f"{make} {model} {year}: {len(results)} complaints found")
    for c in results[:per_vehicle]:
        summary = (c.get("summary") or "").strip()
        if not summary:
            continue
        records.append({
            "id": str(c["odiNumber"]),
            "make": make,
            "model": model,
            "year": year,
            "components": [x.strip() for x in (c.get("components") or "").split(",") if x.strip()],
            "date": c.get("dateComplaintFiled", ""),
            "crash": bool(c.get("crash")),
            "fire": bool(c.get("fire")),
            "injuries": c.get("numberOfInjuries", 0),
            "deaths": c.get("numberOfDeaths", 0),
            "summary": summary,
        })

print("total records:", len(records))

Toyota Camry 2019: 379 complaints found


Honda Accord 2019: 637 complaints found


Tesla Model 3 2021: 644 complaints found


Ford Explorer 2020: 1140 complaints found


Chevrolet Silverado 1500 2020: 683 complaints found
total records: 75


In [3]:
with open("client-complaints.json", "w") as f:
    json.dump(records, f, indent=2)
print("saved", len(records), "records to client-complaints.json")

saved 75 records to client-complaints.json


In [4]:
# Gemini embeddings, 768 dims to match the Pinecone index.
def embed_batch(texts):
    url = f"https://generativelanguage.googleapis.com/v1beta/models/gemini-embedding-001:batchEmbedContents?key={GEMINI_KEY}"
    body = {"requests": [
        {"model": "models/gemini-embedding-001",
         "content": {"parts": [{"text": t}]},
         "outputDimensionality": 768}
        for t in texts
    ]}
    r = requests.post(url, json=body, timeout=120)
    r.raise_for_status()
    return [e["values"] for e in r.json()["embeddings"]]

vectors = []
batch = 25
for i in range(0, len(records), batch):
    chunk = records[i:i+batch]
    embs = embed_batch([r["summary"] for r in chunk])
    for r, emb in zip(chunk, embs):
        meta = {k: r[k] for k in ("make","model","year","components","date","crash","fire","injuries","deaths","summary")}
        vectors.append({"id": r["id"], "values": emb, "metadata": meta})
    print(f"embedded {len(vectors)}/{len(records)}")
    time.sleep(1)

embedded 25/75


embedded 50/75


embedded 75/75


In [5]:
pc = Pinecone(api_key=PINECONE_KEY)
index = pc.Index(INDEX_NAME)
index.upsert(vectors=vectors)
print("upserted", len(vectors), "vectors into", INDEX_NAME)
print(index.describe_index_stats())

upserted 75 vectors into carverdict
{'dimension': 768,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'': {'vector_count': 75}},
 'total_vector_count': 75,
 'vector_type': 'dense'}
